Random output changes every run, so each runner below lists several real runs.

## Popcorn Hack 1: Random SFI Record Number

**Prediction:** a whole number from 1 to 4. `random.randint(1, record_count)` includes **both** endpoints, so 1, 2, 3 and 4 are all possible. Repeats are allowed because each call is independent: getting 3 once does not stop 3 from coming up next time.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Random SFI Record Number
import random

record_count = 4
record_number = random.randint(1, record_count)
print("Selected record:", record_number)

**Actual output (5 real runs, output varies):**

Run 1:

```text
Selected record: 2
```

Run 2:

```text
Selected record: 4
```

Run 3:

```text
Selected record: 3
```

Run 4:

```text
Selected record: 3
```

Run 5:

```text
Selected record: 4
```

## Popcorn Hack 2: Random SFI Part Record

**Prediction:** one whole record. `random.choice(parts)` picks one dictionary, so the name, category and spec always belong together. Picking three separate random fields could mix records up.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - Random SFI Part Record
import random

parts = [
    {"product_name": "Replacement Flywheels and Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"}
]

selected = random.choice(parts)
print("Product:", selected["product_name"])
print("Category:", selected["category"])
print("Spec number:", selected["spec_number"])

**Actual output (3 real runs, output varies):**

Run 1:

```text
Product: Multiple Disc Clutch Assemblies
Category: Drag Racing
Spec number: 1.2
```

Run 2:

```text
Product: Replacement Flywheels and Clutch Assemblies
Category: Auto Racing
Spec number: 1.1
```

Run 3:

```text
Product: Replacement Flywheels and Clutch Assemblies
Category: Auto Racing
Spec number: 1.1
```

## Popcorn Hack 3: Random SFI API Test

**Prediction:** a random part and a random action, routed by `if` / `elif` / `else` to the matching backend-style response.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 - Random SFI API Test
import random

parts = [
    {"product_name": "Replacement Flywheels and Clutch Assemblies", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "spec_number": "2.1"}
]
actions = ["GET/search", "POST/create", "PUT/update"]

part = random.choice(parts)
action = random.choice(actions)
print("Testing", action, "on spec", part["spec_number"])

if action == "GET/search":
    print("200 OK: search returned", part["product_name"])
elif action == "POST/create":
    print("POST: create request sent for", part["product_name"], "(the backend checks for duplicates)")
else:
    print("200 OK: updated", part["product_name"])

**Actual output (3 real runs, output varies):**

Run 1:

```text
Testing PUT/update on spec 1.2
200 OK: updated Multiple Disc Clutch Assemblies
```

Run 2:

```text
Testing PUT/update on spec 2.1
200 OK: updated Racing Flywheel Record
```

Run 3:

```text
Testing POST/create on spec 1.1
POST: create request sent for Replacement Flywheels and Clutch Assemblies (the backend checks for duplicates)
```

## MCQ

1. `random.randint(2, 5)` can return **C** 2, 3, 4, 5. Both endpoints are included.
2. Record A twice in a row is possible because **B** repeats are allowed; each call is independent.
3. The Python for `RANDOM(1, 4)` is **B** `random.randint(1, 4)`.
4. Testing once is not enough because **B** one run shows only one of many possible outcomes.

**MCQ 3.15: 4/4 | answers: C,B,B,B**

## Homework Hack: SFI Backend QA Simulator

`run_backend_test(part, action)` is the reusable backend. `run_qa(test_count)` loops `test_count` times, picks a whole record and an action at random, and prints the record, the action and the result.

**All possible outputs:** 3 parts × 4 actions = 12 combinations, and each result depends on whether the spec is currently stored:

| Action | Spec stored | Spec not stored |
|---|---|---|
| GET/search | 200 found | 404 not found |
| POST/create | 409 REJECTED duplicate | 201 created (added to the stored list) |
| PUT/update | 200 updated | 404 cannot update |
| DELETE/remove | 200 removed (taken off the list) | 404 cannot remove |

Because create and remove change `existing_spec_numbers`, a later test can get a different result for the same part and action.

In [ ]:
# CODE_RUNNER: Homework Hack - SFI Backend QA Simulator
import random

parts = [
    {"product_name": "Replacement Flywheels and Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"}
]

actions = ["GET/search", "POST/create", "PUT/update", "DELETE/remove"]

existing_spec_numbers = ["1.1", "2.1"]
test_count = 5

def run_backend_test(part, action):
    spec = part["spec_number"]
    if action == "GET/search":
        if spec in existing_spec_numbers:
            return "200 OK: found record " + spec
        else:
            return "404 Not Found: no record " + spec
    elif action == "POST/create":
        if spec in existing_spec_numbers:
            return "409 Conflict: REJECTED, duplicate spec " + spec
        else:
            existing_spec_numbers.append(spec)
            return "201 Created: record " + spec
    elif action == "PUT/update":
        if spec in existing_spec_numbers:
            return "200 OK: updated record " + spec
        else:
            return "404 Not Found: cannot update " + spec
    else:
        if spec in existing_spec_numbers:
            existing_spec_numbers.remove(spec)
            return "200 OK: removed record " + spec
        else:
            return "404 Not Found: cannot remove " + spec

def run_qa(test_count):
    for run in range(1, test_count + 1):
        part = random.choice(parts)       # the whole record, so its fields stay together
        action = random.choice(actions)
        print("Test", run, "|", action, "|", part["product_name"], "(" + part["category"] + ", spec " + part["spec_number"] + ")")
        print("   ->", run_backend_test(part, action))
    print("Stored specs after testing:", existing_spec_numbers)

run_qa(test_count)

**Actual output (3 real runs, output varies):**

Run 1:

```text
Test 1 | GET/search | Multiple Disc Clutch Assemblies (Drag Racing, spec 1.2)
   -> 404 Not Found: no record 1.2
Test 2 | PUT/update | Replacement Flywheels and Clutch Assemblies (Auto Racing, spec 1.1)
   -> 200 OK: updated record 1.1
Test 3 | DELETE/remove | Racing Flywheel Record (Auto Racing, spec 2.1)
   -> 200 OK: removed record 2.1
Test 4 | GET/search | Replacement Flywheels and Clutch Assemblies (Auto Racing, spec 1.1)
   -> 200 OK: found record 1.1
Test 5 | DELETE/remove | Racing Flywheel Record (Auto Racing, spec 2.1)
   -> 404 Not Found: cannot remove 2.1
Stored specs after testing: ['1.1']
```

Run 2:

```text
Test 1 | PUT/update | Replacement Flywheels and Clutch Assemblies (Auto Racing, spec 1.1)
   -> 200 OK: updated record 1.1
Test 2 | DELETE/remove | Multiple Disc Clutch Assemblies (Drag Racing, spec 1.2)
   -> 404 Not Found: cannot remove 1.2
Test 3 | POST/create | Multiple Disc Clutch Assemblies (Drag Racing, spec 1.2)
   -> 201 Created: record 1.2
Test 4 | PUT/update | Multiple Disc Clutch Assemblies (Drag Racing, spec 1.2)
   -> 200 OK: updated record 1.2
Test 5 | DELETE/remove | Multiple Disc Clutch Assemblies (Drag Racing, spec 1.2)
   -> 200 OK: removed record 1.2
Stored specs after testing: ['1.1', '2.1']
```

Run 3:

```text
Test 1 | PUT/update | Replacement Flywheels and Clutch Assemblies (Auto Racing, spec 1.1)
   -> 200 OK: updated record 1.1
Test 2 | PUT/update | Multiple Disc Clutch Assemblies (Drag Racing, spec 1.2)
   -> 404 Not Found: cannot update 1.2
Test 3 | POST/create | Multiple Disc Clutch Assemblies (Drag Racing, spec 1.2)
   -> 201 Created: record 1.2
Test 4 | POST/create | Racing Flywheel Record (Auto Racing, spec 2.1)
   -> 409 Conflict: REJECTED, duplicate spec 2.1
Test 5 | POST/create | Multiple Disc Clutch Assemblies (Drag Racing, spec 1.2)
   -> 409 Conflict: REJECTED, duplicate spec 1.2
Stored specs after testing: ['1.1', '2.1', '1.2']
```

### Reproducible documented run

`random.seed(15)` makes the random choices the same every time, so this run can be checked against its recorded output.

In [ ]:
# CODE_RUNNER: Homework Hack test - seeded run (same result every time)
import random

parts = [
    {"product_name": "Replacement Flywheels and Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"}
]

actions = ["GET/search", "POST/create", "PUT/update", "DELETE/remove"]

existing_spec_numbers = ["1.1", "2.1"]
test_count = 5

def run_backend_test(part, action):
    spec = part["spec_number"]
    if action == "GET/search":
        if spec in existing_spec_numbers:
            return "200 OK: found record " + spec
        else:
            return "404 Not Found: no record " + spec
    elif action == "POST/create":
        if spec in existing_spec_numbers:
            return "409 Conflict: REJECTED, duplicate spec " + spec
        else:
            existing_spec_numbers.append(spec)
            return "201 Created: record " + spec
    elif action == "PUT/update":
        if spec in existing_spec_numbers:
            return "200 OK: updated record " + spec
        else:
            return "404 Not Found: cannot update " + spec
    else:
        if spec in existing_spec_numbers:
            existing_spec_numbers.remove(spec)
            return "200 OK: removed record " + spec
        else:
            return "404 Not Found: cannot remove " + spec

def run_qa(test_count):
    for run in range(1, test_count + 1):
        part = random.choice(parts)       # the whole record, so its fields stay together
        action = random.choice(actions)
        print("Test", run, "|", action, "|", part["product_name"], "(" + part["category"] + ", spec " + part["spec_number"] + ")")
        print("   ->", run_backend_test(part, action))
    print("Stored specs after testing:", existing_spec_numbers)

random.seed(15)
run_qa(test_count)

**Actual output:**

```text
Test 1 | GET/search | Replacement Flywheels and Clutch Assemblies (Auto Racing, spec 1.1)
   -> 200 OK: found record 1.1
Test 2 | GET/search | Racing Flywheel Record (Auto Racing, spec 2.1)
   -> 200 OK: found record 2.1
Test 3 | POST/create | Replacement Flywheels and Clutch Assemblies (Auto Racing, spec 1.1)
   -> 409 Conflict: REJECTED, duplicate spec 1.1
Test 4 | GET/search | Replacement Flywheels and Clutch Assemblies (Auto Racing, spec 1.1)
   -> 200 OK: found record 1.1
Test 5 | POST/create | Racing Flywheel Record (Auto Racing, spec 2.1)
   -> 409 Conflict: REJECTED, duplicate spec 2.1
Stored specs after testing: ['1.1', '2.1']
```

### Required POST/create check

Not random: POST/create on every part. 1.1 and 2.1 must be rejected as duplicates and 1.2 must be accepted.

In [ ]:
# CODE_RUNNER: Homework Hack test - POST/create rejects duplicates, accepts 1.2
import random

parts = [
    {"product_name": "Replacement Flywheels and Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"}
]

actions = ["GET/search", "POST/create", "PUT/update", "DELETE/remove"]

existing_spec_numbers = ["1.1", "2.1"]
test_count = 5

def run_backend_test(part, action):
    spec = part["spec_number"]
    if action == "GET/search":
        if spec in existing_spec_numbers:
            return "200 OK: found record " + spec
        else:
            return "404 Not Found: no record " + spec
    elif action == "POST/create":
        if spec in existing_spec_numbers:
            return "409 Conflict: REJECTED, duplicate spec " + spec
        else:
            existing_spec_numbers.append(spec)
            return "201 Created: record " + spec
    elif action == "PUT/update":
        if spec in existing_spec_numbers:
            return "200 OK: updated record " + spec
        else:
            return "404 Not Found: cannot update " + spec
    else:
        if spec in existing_spec_numbers:
            existing_spec_numbers.remove(spec)
            return "200 OK: removed record " + spec
        else:
            return "404 Not Found: cannot remove " + spec

def run_qa(test_count):
    for run in range(1, test_count + 1):
        part = random.choice(parts)       # the whole record, so its fields stay together
        action = random.choice(actions)
        print("Test", run, "|", action, "|", part["product_name"], "(" + part["category"] + ", spec " + part["spec_number"] + ")")
        print("   ->", run_backend_test(part, action))
    print("Stored specs after testing:", existing_spec_numbers)

for part in parts:
    print(part["spec_number"], "->", run_backend_test(part, "POST/create"))
print("Stored specs:", existing_spec_numbers)

**Actual output:**

```text
1.1 -> 409 Conflict: REJECTED, duplicate spec 1.1
1.2 -> 201 Created: record 1.2
2.1 -> 409 Conflict: REJECTED, duplicate spec 2.1
Stored specs: ['1.1', '2.1', '1.2']
```

## Submission Notes

```text
Lesson: CSP 3.15 Random Values
Notebook: https://rydenb.github.io/portfolio/python/random-hw
MCQ 3.15: 4/4 | answers: C,B,B,B
Popcorn Hacks 1–3 completed during class and copied to notebook: yes
Homework tested: 3 unseeded runs of 5 tests each (results vary), 1 seeded run (random.seed(15)), and a POST/create check: 1.1 -> 409 REJECTED duplicate, 1.2 -> 201 Created, 2.1 -> 409 REJECTED duplicate.
Seeded run output:
Test 1 | GET/search | Replacement Flywheels and Clutch Assemblies (Auto Racing, spec 1.1)
   -> 200 OK: found record 1.1
Test 2 | GET/search | Racing Flywheel Record (Auto Racing, spec 2.1)
   -> 200 OK: found record 2.1
Test 3 | POST/create | Replacement Flywheels and Clutch Assemblies (Auto Racing, spec 1.1)
   -> 409 Conflict: REJECTED, duplicate spec 1.1
Test 4 | GET/search | Replacement Flywheels and Clutch Assemblies (Auto Racing, spec 1.1)
   -> 200 OK: found record 1.1
Test 5 | POST/create | Racing Flywheel Record (Auto Racing, spec 2.1)
   -> 409 Conflict: REJECTED, duplicate spec 2.1
Stored specs after testing: ['1.1', '2.1']
```